# Word-Level LSTM Text Generation - Comparative Experiments
**Task:** Develop a text generator using Long Short-Term Memory (LSTM) recurrent neural networks, trained on Shakespeare's Complete Works using **word-level tokenization**.

## Architecture & Workflow Overview
```text
Shakespeare Complete Works -> Preprocessing (lowercase + remove punct) -> Word Tokenization -> 
Vocabulary (<PAD>=0, <UNK>=1) -> Sequence Window (30 words -> next word) -> 90/10 Train/Val Split -> 
Embedding (128-dim) -> Stacked LSTM (256-dim, 2 layers, 0.2 dropout) -> Linear Output -> 
CrossEntropyLoss / Softmax -> Temperature Sampling (0.5, 0.8, 1.0) -> Generated Text
```

In [ ]:
# Step 0: Environment Setup and Modular Imports
import os
import sys
import torch
import matplotlib.pyplot as plt

os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
sys.path.append('../src')

from preprocess import download_dataset, load_and_clean_text, build_vocabulary
from dataset import prepare_data
from model import LSTMTextGenerator
from train import run_training, plot_metrics
from generate import generate_text, load_model_checkpoint, generate_sample_suite
from experiments import run_all_experiments

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Compute Device: {device}')

## Step 1: Preprocessing & Vocabulary Inspection
We clean Gutenberg headers, lowercase text, strip punctuation, tokenize words, and reserve `<PAD>` (0) and `<UNK>` (1).

In [ ]:
# Download & Inspect Preprocessed Text
data_path = download_dataset(data_dir='../data')
cleaned_text = load_and_clean_text(data_path)
words = cleaned_text.split()
print(f'Total Corpus Words: {len(words):,}')
print('Sample Text: ' + ' '.join(words[:25]))

# Build DataLoaders with 90/10 Split and Sequence Length = 30 words
train_loader, val_loader, metadata = prepare_data(data_path=data_path, seq_length=30, batch_size=64, val_split=0.1)
print(f'Vocabulary Size (with <PAD> & <UNK>): {metadata["vocab_size"]:,}')
print(f'Train Sequences: {metadata["train_size"]:,}, Val Sequences: {metadata["val_size"]:,}')

## Step 2: Model Architecture
The PyTorch model maps sequence input IDs -> Embedding (128-dim) -> Stacked LSTM (2 layers, 256-dim, 0.2 dropout) -> Linear projection (12,850 vocab size).

In [ ]:
vocab_size = metadata['vocab_size']
model = LSTMTextGenerator(
    vocab_size=vocab_size,
    embedding_dim=128,
    hidden_dim=256,
    num_layers=2,
    dropout=0.2
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'Total Trainable Parameters: {total_params:,}')

## Step 3: Model Training with Early Stopping & Perplexity Tracking
We train using Adam (`lr=0.001`) and CrossEntropyLoss. EarlyStopping tracks validation loss and saves the best model checkpoint to `models/best_model.pt`.

In [ ]:
trained_model, metadata, history = run_training(
    epochs=5,
    seq_length=30,
    batch_size=64,
    learning_rate=0.001,
    embedding_dim=128,
    hidden_dim=256,
    num_layers=2,
    patience=3,
    save_path='../models/best_model.pt',
    history_plot_path='../outputs/training_loss.png'
)

## Step 4: Iterative Text Generation & Temperature Sampling
We generate new text by feeding word seed prompts (`"to be or not"`, `"the king"`, `"my lord"`) and iteratively predicting the next word using Temperature sampling ($T=0.5, 0.8, 1.0$).

In [ ]:
seeds = ['to be or not', 'the king', 'my lord', 'shall i compare']

for seed in seeds:
    print(f'\n========================================')
    print(f' Seed Prompt: "{seed}"')
    print(f'========================================')
    for temp in [0.5, 0.8, 1.0]:
        generated = generate_text(
            model=trained_model,
            metadata=metadata,
            seed_text=seed,
            num_words=60,
            temperature=temp,
            device=device
        )
        print(f'\n[Temperature {temp}]:')
        print(generated)
        print('-' * 50)

## Step 5: Bonus Controlled Architecture Experiments
We evaluate 3 controlled configurations:
1. **Baseline**: 1 Layer LSTM, Sequence Length = 20
2. **Experiment A**: 2 Layer Stacked LSTM, Sequence Length = 20
3. **Experiment B**: 2 Layer Stacked LSTM, Sequence Length = 40

In [ ]:
# Run controlled architecture benchmark suite
run_all_experiments(output_dir='../outputs')